In [1]:
from pathlib import Path
from tqdm import tqdm
import yaml
import pandas as pd
import numpy as np

In [2]:
import torch
from torch.utils.data import DataLoader, Subset

In [3]:
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, QuantFormat, QuantType, CalibrationDataReader

In [4]:
from rtal.datasets.dataset import ROMDataset
from mlp_for_quantization import MLP

In [5]:
onnx_folder = Path('onnx_files_narrow')
device = 'cpu'

In [9]:
# gpuserver0
data_root = '/data/yhuang2/rtal/rom_det-3_part-200_cont-and-rounded'

# flora
data_root = '/data/rtal/rom_det-3_part-200_cont-and-rounded'

# calibration dataset first [num_samples] of the training dataset
num_samples = 2000
dataset_cali = ROMDataset(data_root, split='train', num_particles=50)
cali_bs = 4
dataloader_cali = DataLoader(Subset(dataset_cali, range(num_samples)), batch_size=cali_bs, shuffle=False)
print(f'Number of cali batches (bs={cali_bs}), {len(dataloader_cali)}')

# test dataset for accuracy
dataset_test = ROMDataset(data_root, split='test', num_particles=50)
test_bs = 1
dataloader_test = DataLoader(dataset_test, batch_size=test_bs, shuffle=False)
print(f'Number of test batches (bs={test_bs}), {len(dataloader_test)}')

Number of cali batches (bs=4), 500
Number of test batches (bs=1), 10000


In [10]:
with open('checkpoints/config_narrow.yaml', 'r', encoding='UTF-8') as handle:
    config = yaml.safe_load(handle)

# wrap the MLP model inside a model with quant/dequant stubs
model = MLP(**config['model'])

FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/config_narrow.yaml'

In [7]:
ckpt_path = 'checkpoints/ckpt_last_narrow.pth'
ckpt = torch.load(ckpt_path, map_location='cpu')
model_state_dict = ckpt['model']

# load the pretrained weights into the model
model.load_state_dict(model_state_dict)
model.eval()

MLP(
  (embed): Sequential(
    (0): Identity()
    (1): Linear(in_features=6, out_features=128, bias=True)
    (2): LeakyReLU(negative_slope=0.1)
    (3): Identity()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): LeakyReLU(negative_slope=0.1)
  )
  (solvers): ModuleList(
    (0-2): 3 x SubsetSolver(
      (model): Sequential(
        (0): Identity()
        (1): Linear(in_features=256, out_features=128, bias=True)
        (2): LeakyReLU(negative_slope=0.1)
        (3): LinearBlock(
          (norm_layer): Identity()
          (linear): Linear(in_features=128, out_features=128, bias=True)
          (activ): LeakyReLU(negative_slope=0.1)
        )
        (4): LinearBlock(
          (norm_layer): Identity()
          (linear): Linear(in_features=128, out_features=128, bias=True)
          (activ): LeakyReLU(negative_slope=0.1)
        )
        (5): LinearBlock(
          (norm_layer): Identity()
          (linear): Linear(in_features=128, out_features=128, bias=

In [17]:
onnx_path = onnx_folder/f'mlp_fp32.onnx'
model_input = torch.randn(1, 50, 6)

torch.onnx.export(
    model,                               # model being run
    model_input,                         # model input (or a tuple for multiple inputs)
    onnx_path,                           # where to save the model (filename)
    export_params       = True,          # store the trained weights inside the model
    opset_version       = 11,            # the ONNX version to export to (11 is widely supported)
    do_constant_folding = True,          # optimize constants
    input_names         = ['input'],     # input name (can be arbitrary)
    output_names        = ['output'],    # output name
    # support dynamic batch size
    dynamic_axes        = {'input'  : {0: 'batch_size'},
                           'output' : {0: 'batch_size'}}
)

In [19]:
# Define a CalibrationDataReader (replace with your actual data handling)
class CalibrationDataReader(CalibrationDataReader):
    def __init__(self, 
                 dataloader, 
                 input_name="input", 
                 device="cpu"):
        
        self.dataloader = dataloader
        self.input_name = input_name
        self.device = device

        # Load everything into memory for calibration
        self.data_iter = iter(self.dataloader)
        self.cache = []
        
        for event in self.dataloader:
            # to numpy float32
            readout = event[f'readout_curr_cont']
            readout = torch.transpose(readout, 1, 2).flatten(-2, -1)
            readout = readout.to(self.device).float().cpu().numpy()
            
            self.cache.append({self.input_name: readout})

        self.index = 0

    def get_next(self):
        if self.index < len(self.cache):
            item = self.cache[self.index]
            self.index += 1
            return item
        return None

# Instantiate the data reader
dr = CalibrationDataReader(dataloader_cali)

input_model_path = onnx_folder/"mlp_fp32.onnx"
output_model_path = onnx_folder/"mlp_int8.onnx"

# Perform static quantization
quantize_static(
    input_model_path,
    output_model_path,
    dr,
    quant_format=QuantFormat.QDQ, # Recommended format for better performance and accuracy
    per_channel=False,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8
)
print(f"Quantized INT8 model saved to {output_model_path}")

Quantized INT8 model saved to onnx_files_narrow/mlp_int8.onnx


# compare the raw and quantized model

In [23]:
FLOAT_MODEL_PATH = "mlp_fp32.onnx"
INT8_MODEL_PATH  = "mlp_int8.onnx"

sess_fp32 = ort.InferenceSession(onnx_folder/"mlp_fp32.onnx", providers=["CPUExecutionProvider"])
sess_int8 = ort.InferenceSession(onnx_folder/"mlp_int8.onnx", providers=["CPUExecutionProvider"])

In [28]:
def ort_predict(session, x):
    """
    Run ONNX model on a PyTorch tensor x.
    """
    
    input_name = session.get_inputs()[0].name
    outputs = session.run(None, {input_name: x.numpy()})
    
    return outputs[0]

In [30]:
stat = {'max_diff': [], 
        'l1_error': [], 
        'l1_norm': []}

with torch.no_grad():
    for event in tqdm(dataloader_test):
        
        readout = event[f'readout_curr_cont'].to(device)
        readout = torch.transpose(readout, 1, 2).flatten(-2, -1)
        
        output_full = ort_predict(sess_fp32, readout)
        output_int8 = ort_predict(sess_int8, readout)

        diff = np.abs(output_full - output_int8)
        
        stat['max_diff'].append(diff.max())
        stat['l1_error'].append(diff.mean())
        stat['l1_norm'].append(np.abs(output_full).mean())

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [00:45<00:00, 221.16it/s]


NameError: name 'pd' is not defined

In [39]:
df = pd.DataFrame(data=stat)
df.to_csv('results/float_int8_diff_onnx.csv', index=False)

In [38]:
df

,max_diff,l1_error,l1_norm
0,0.023150,0.005682,0.025564
1,0.040390,0.007335,0.046194
2,0.057054,0.011222,0.026076
3,0.015193,0.005032,0.030572
4,0.050728,0.009941,0.015759
...,...,...,...
9995,0.035095,0.005770,0.031675
9996,0.027011,0.006473,0.015807
9997,0.046142,0.009668,0.028360
9998,0.029294,0.005976,0.040203
